In [38]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sqlalchemy import create_engine

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [39]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "customer_revenue_platform"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Connected Successfully")

Connected Successfully


In [40]:
query = """
SELECT *
FROM customer_features
"""

customer_features = pd.read_sql(
    query,
    engine
)

In [41]:
customer_features.shape
customer_features.head()
customer_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   customer_unique_id      96096 non-null  str           
 1   total_orders            96096 non-null  int64         
 2   total_revenue           96096 non-null  float64       
 3   avg_order_value         96096 non-null  float64       
 4   avg_review_score        96096 non-null  float64       
 5   total_freight           96096 non-null  float64       
 6   avg_installments        96096 non-null  float64       
 7   first_purchase          96096 non-null  datetime64[us]
 8   last_purchase           96096 non-null  datetime64[us]
 9   customer_lifetime_days  96096 non-null  int64         
 10  recency_days            96096 non-null  int64         
 11  repeat_customer         96096 non-null  int64         
 12  customer_tenure_months  96096 non-null  float64       
 1

In [42]:
customer_features.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_revenue',
 'avg_order_value',
 'avg_review_score',
 'total_freight',
 'avg_installments',
 'first_purchase',
 'last_purchase',
 'customer_lifetime_days',
 'recency_days',
 'repeat_customer',
 'customer_tenure_months',
 'revenue_per_day',
 'high_value_customer',
 'freight_percentage',
 'review_category',
 'recency_group',
 'R_score',
 'F_score',
 'M_score',
 'RFM_score',
 'customer_value_score']

In [43]:
# target variable
y = customer_features["repeat_customer"]

In [44]:
# checking for repeat customers
customer_features["repeat_customer"].value_counts()

repeat_customer
0    93099
1     2997
Name: count, dtype: int64

In [45]:
# checking for repeat customers in percentage
customer_features["repeat_customer"].value_counts(normalize=True) * 100

repeat_customer
0    96.881244
1     3.118756
Name: proportion, dtype: float64

In [46]:
# checking for missng values
customer_features.isnull().sum()

customer_unique_id          0
total_orders                0
total_revenue               0
avg_order_value             0
avg_review_score            0
total_freight               0
avg_installments            0
first_purchase              0
last_purchase               0
customer_lifetime_days      0
recency_days                0
repeat_customer             0
customer_tenure_months      0
revenue_per_day             0
high_value_customer         0
freight_percentage          0
review_category           716
recency_group               2
R_score                     0
F_score                     0
M_score                     0
RFM_score                   0
customer_value_score        0
dtype: int64

In [47]:
# checking for repeat customers
customer_features["repeat_customer"].value_counts()

customer_features["repeat_customer"].value_counts(normalize=True) * 100

customer_features.isnull().sum()

customer_unique_id          0
total_orders                0
total_revenue               0
avg_order_value             0
avg_review_score            0
total_freight               0
avg_installments            0
first_purchase              0
last_purchase               0
customer_lifetime_days      0
recency_days                0
repeat_customer             0
customer_tenure_months      0
revenue_per_day             0
high_value_customer         0
freight_percentage          0
review_category           716
recency_group               2
R_score                     0
F_score                     0
M_score                     0
RFM_score                   0
customer_value_score        0
dtype: int64

In [48]:
# 
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [49]:
customer_features["review_category"] = (
    customer_features["review_category"]
    .fillna("Unknown")
)

customer_features["recency_group"] = (
    customer_features["recency_group"]
    .fillna("Unknown")
)

In [50]:
customer_features.isnull().sum().sum()

np.int64(0)

In [51]:
y = customer_features["repeat_customer"]

In [52]:
leakage_columns = [
    "total_orders",
    "F_score",
    "RFM_score",
    "customer_value_score"
]

In [53]:
X = customer_features.drop(
    columns=[
        "customer_unique_id",
        "repeat_customer",
        "first_purchase",
        "last_purchase"
    ] + leakage_columns
)

In [54]:
X = pd.get_dummies(
    X,
    drop_first=True
)

In [55]:
import joblib

joblib.dump(
    X.columns.tolist(),
    "../trained_models/purchase_prediction/feature_columns.pkl"
)

['../trained_models/purchase_prediction/feature_columns.pkl']

In [56]:
import numpy as np

X.replace(
    [np.inf, -np.inf],
    0,
    inplace=True
)

,total_revenue,avg_order_value,avg_review_score,total_freight,avg_installments,customer_lifetime_days,recency_days,customer_tenure_months,revenue_per_day,high_value_customer,...,recency_group_Very Recent,recency_group_Warm,R_score_2,R_score_3,R_score_4,R_score_5,M_score_2,M_score_3,M_score_4,M_score_5
0,141.90,141.90,5.0,12.00,8.0,0,160,0.0,141.90,0,...,False,True,False,False,True,False,False,False,True,False
1,27.19,27.19,4.0,8.29,1.0,0,163,0.0,27.19,0,...,False,True,False,False,True,False,False,False,False,False
2,86.22,86.22,3.0,17.22,8.0,0,585,0.0,86.22,0,...,False,False,False,False,False,False,True,False,False,False
3,43.62,43.62,4.0,17.63,4.0,0,369,0.0,43.62,0,...,False,False,True,False,False,False,False,False,False,False
4,196.89,196.89,5.0,16.89,6.0,0,336,0.0,196.89,0,...,False,False,True,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96091,4134.84,2067.42,5.0,497.42,10.0,0,495,0.0,4134.84,1,...,False,False,False,False,False,False,False,False,False,True
96092,84.58,84.58,4.0,19.69,1.0,0,310,0.0,84.58,0,...,False,False,False,True,False,False,True,False,False,False
96093,112.46,112.46,5.0,22.56,1.0,0,617,0.0,112.46,0,...,False,False,False,False,False,False,False,True,False,False
96094,133.69,133.69,5.0,18.69,5.0,0,168,0.0,133.69,0,...,False,True,False,False,True,False,False,True,False,False


In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [58]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [59]:
scale_pos_weight = (
    y_train.value_counts()[0]
    /
    y_train.value_counts()[1]
)

print(scale_pos_weight)

31.058381984987488


In [60]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(
    X_train,
    y_train
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [61]:
from sklearn.metrics import classification_report

y_pred = xgb_model.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.99      1.00      1.00     18621
           1       0.95      0.78      0.86       599

    accuracy                           0.99     19220
   macro avg       0.97      0.89      0.93     19220
weighted avg       0.99      0.99      0.99     19220



In [62]:
classification_report(y_test, y_pred)

'              precision    recall  f1-score   support\n\n           0       0.99      1.00      1.00     18621\n           1       0.95      0.78      0.86       599\n\n    accuracy                           0.99     19220\n   macro avg       0.97      0.89      0.93     19220\nweighted avg       0.99      0.99      0.99     19220\n'

In [63]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[18598    23]
 [  131   468]]


In [64]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

Accuracy : 0.991987513007284
Precision: 0.9531568228105907
Recall   : 0.7813021702838063
F1 Score : 0.8587155963302753


In [65]:
train_pred =xgb_model.predict(X_train)

print(
    "Train Accuracy:",
    accuracy_score(y_train, train_pred)
)

print(
    "Test Accuracy:",
    accuracy_score(y_test, y_pred)
)

Train Accuracy: 0.9956033092252459
Test Accuracy: 0.991987513007284


In [66]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_model.feature_importances_
})

importance.sort_values(
    by="importance",
    ascending=False
).head(20)

,feature,importance
5,customer_lifetime_days,0.880348
7,customer_tenure_months,0.058896
3,total_freight,0.006475
1,avg_order_value,0.005619
8,revenue_per_day,0.005466
0,total_revenue,0.005239
2,avg_review_score,0.003589
23,M_score_2,0.003397
4,avg_installments,0.002776
12,review_category_Poor,0.002499


In [67]:
prob = xgb_model.predict_proba(X_test)

prob[:5]

array([[9.9998868e-01, 1.1337964e-05],
       [9.9991739e-01, 8.2601066e-05],
       [9.9994898e-01, 5.1035044e-05],
       [9.9909413e-01, 9.0586854e-04],
       [9.9997783e-01, 2.2180300e-05]], dtype=float32)

In [68]:
from sklearn.metrics import roc_auc_score

prob = xgb_model.predict_proba(X_test)[:,1]

roc_auc_score(
    y_test,
    prob
)

0.9932117498159178

In [69]:
model_results = pd.DataFrame({
    "Model": ["XGBOOST"],
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test,y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC_AUC": roc_auc_score(y_test, prob),
})

model_results

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,XGBOOST,0.991988,0.953157,0.781302,0.858716,0.993212


In [70]:
# STORING ACCURACY, PRECISION, RECALL, F1 SCORE AND ROC AUC OF XGBOOST IN POSTGRESQL DATABASE
from datetime import datetime

xgb_results = pd.DataFrame({
    "model_name": ["XGBoost"],
    "accuracy": [accuracy_score(y_test, y_pred)],
    "precision": [precision_score(y_test, y_pred)],
    "recall": [recall_score(y_test, y_pred)],
    "f1_score": [f1_score(y_test, y_pred)],
    "roc_auc": [roc_auc_score(y_test, prob)],
    "run_date": [datetime.now()]
})

from sqlalchemy import text

model_name = "XGBoost"

with engine.begin() as conn:
    conn.execute(
        text("DELETE FROM model_metrics WHERE model_name = :model"),
        {"model": model_name}
    )

xgb_results.to_sql(
    "model_metrics",
    con=engine,
    if_exists="append",
    index=False
)

1

In [33]:
# SAVING THE XGBOOST MODEL TO A FILE
import joblib

joblib.dump(
    xgb_model,
    "../trained_models/purchase_prediction/xgboost.pkl"
)

['../trained_models/purchase_prediction/xgboost.pkl']

In [35]:
# storing the data into postgress database
y_pred = xgb_model.predict(X)
y_prob = xgb_model.predict_proba(X)[:,1]

In [74]:
prediction_results = pd.DataFrame({
    "customer_unique_id": customer_features["customer_unique_id"].values,
    "prediction": y_pred_all,
    "purchase_probability": y_prob
})

In [75]:
prediction_results.to_sql(
    "xgboost_predictions",
    con=engine,
    if_exists="replace",
    index=False
)

96